# Module Economie - Preparation et harmonisation

Ce notebook nettoie les jeux de donnees Economie, harmonise les colonnes et normalise la dimension temporelle.

Regles temporelles appliquees :
- `YYYYQn` ou `YYYY Qn` devient `YYYY Tn`
- `YYYYMn` ou `YYYY Mn` devient `mois annee` (ex: `2016M1` -> `janvier 2016`)
- `YYYY` reste une periode annuelle

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
RAW_ROOT = ROOT / 'data_raw' / 'economie_emploi'
OUT_ROOT = ROOT / 'data' / 'Economie'

OUT_DIRS = {
    'activites_economiques': OUT_ROOT / 'Activites_economiques',
    'emploi': OUT_ROOT / 'Emploi',
    'entreprises': OUT_ROOT / 'Entreprises',
    'prix': OUT_ROOT / 'Prix',
}

for d in OUT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

RAW_GROUPS = {
    'activites_economiques': RAW_ROOT / 'activites_economiques_locales',
    'emploi': RAW_ROOT / 'emploi_chomage',
    'entreprises': RAW_ROOT / 'pme_entreprises',
    'prix': RAW_ROOT / 'prix_denrees',
}

for k, p in RAW_GROUPS.items():
    print(f'- {k}: {p}')

In [ ]:
FRENCH_MONTHS = {
    1: 'janvier', 2: 'fevrier', 3: 'mars', 4: 'avril', 5: 'mai', 6: 'juin',
    7: 'juillet', 8: 'aout', 9: 'septembre', 10: 'octobre', 11: 'novembre', 12: 'decembre'
}

PHRASES_REMPLACEMENTS_FR = {
    'burkina faso': 'Burkina Faso',
    'ensemble': 'Ensemble',
    'market average': 'Moyenne marche',
}

COLUMN_RENAMES_FR = {
    'country': 'pays',
    'adm1_name': 'region',
    'adm2_name': 'province',
    'admin1': 'region',
    'admin2': 'province',
    'mkt_name': 'marche',
    'market': 'marche',
    'category': 'categorie',
    'commodity': 'produit',
    'priceflag': 'statut_prix',
    'pricetype': 'type_prix',
    'currency': 'devise',
    'components': 'composantes',
    'dates': 'date_iso',
    'year': 'annee_source',
    'month': 'mois_source',
    'price': 'prix_local',
    'usdprice': 'prix_usd',
    'beans': 'haricots',
    'maize': 'mais',
    'maize_fao': 'maïs_fao',
    'millet': 'mil',
    'millet_fao': 'mil_fao',
    'rice_fao': 'riz_fao',
    'sorghum': 'sorgho',
    'sorghum_fao': 'sorgho_fao',
}

VALUE_REMAP_BY_COLUMN = {
    'type_prix': {
        'Retail': 'detail',
        'Wholesale': 'gros',
    },
    'statut_prix': {
        'actual': 'observe',
        'estimated': 'estime',
        'forecast': 'prevision',
    },
    'categorie': {
        'cereals and tubers': 'cereales et tubercules',
        'non-food': 'hors alimentaire',
        'oil and fats': 'huiles et matieres grasses',
        'meat, fish and eggs': 'viande, poisson et oeufs',
        'milk and dairy': 'lait et produits laitiers',
        'pulses and nuts': 'legumineuses et fruits a coque',
    },
    'region': {
        'Market Average': 'Moyenne marche',
    },
    'province': {
        'Market Average': 'Moyenne marche',
    },
    'marche': {
        'Market Average': 'Moyenne marche',
    },
}

WFP_WORD_REMAP_FR = {
    'beans': 'haricots',
    'maize': 'mais',
    'maize_fao': 'maïs_fao',
    'index weight': "poids de l'indice",
    'millet': 'mil',
    'millet_fao': 'mil_fao',
    'rice': 'riz',
    'rice_fao': 'riz_fao',
    'sorghum': 'sorgho',
    'sorghum_fao': 'sorgho_fao',
    'wheat': 'ble',
    'flour': 'farine',
    'bread': 'pain',
    'potatoes': 'pommes de terre',
    'onions': 'oignons',
    'tomatoes': 'tomates',
    'sugar': 'sucre',
    'oil': 'huile',
    'cassava': 'manioc',
    'eggplants': 'aubergines',
    'fish': 'poisson',
    'meat': 'viande',
}

UNNECESSARY_COLUMNS = {
    'lat', 'lon', 'latitude', 'longitude',
    'market_id', 'commodity_id', 'geo_id',
    'iso3', 'countryiso3',
    'index_confidence_score', 'spatially_interpolated',
    'start_dense_data', 'last_survey_point',
    'data_coverage', 'data_coverage_recent',
}

UNNECESSARY_PREFIXES = (
    'o_', 'h_', 'l_', 'c_', 'inflation_', 'trust_'
)

def normalize_columns(columns):
    clean = []
    seen = {}
    for col in columns:
        s = str(col).strip().lower()
        s = s.replace("'", '')
        s = re.sub(r'[^a-z0-9]+', '_', s)
        s = re.sub(r'_+', '_', s).strip('_')

        if s in seen:
            seen[s] += 1
            s = f"{s}_{seen[s]}"
        else:
            seen[s] = 0

        clean.append(s)
    return clean

def normalize_text(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def translate_text_fr(value):
    if pd.isna(value):
        return value
    s = str(value)
    s_low = s.strip().lower()
    if s_low in PHRASES_REMPLACEMENTS_FR:
        return PHRASES_REMPLACEMENTS_FR[s_low]
    return s

def coalesce_duplicate_columns(df):
    result = pd.DataFrame(index=df.index)
    for col in df.columns:
        if col in result.columns:
            result[col] = result[col].combine_first(df[col])
        else:
            result[col] = df[col]
    return result

def rename_columns_fr(df):
    renamed = [COLUMN_RENAMES_FR.get(col, col) for col in df.columns]
    df.columns = renamed
    return coalesce_duplicate_columns(df)

def translate_wfp_text(value):
    if pd.isna(value):
        return value
    s = str(value)
    for en, fr in WFP_WORD_REMAP_FR.items():
        if '_' in en:
            s = s.replace(en, fr)
        else:
            s = re.sub(rf'\b{re.escape(en)}\b', fr, s, flags=re.IGNORECASE)
    return s

def translate_values_by_column(df):
    for col, mapping in VALUE_REMAP_BY_COLUMN.items():
        if col in df.columns:
            df[col] = df[col].replace(mapping)

    for col in ['categorie', 'produit', 'composantes']:
        if col in df.columns:
            df[col] = df[col].map(translate_wfp_text)

    return df

def drop_fully_empty_rows(df):
    before = len(df)
    df2 = df.dropna(how='all').copy()
    return df2, before - len(df2)

def parse_time_value(raw):
    if isinstance(raw, pd.Series):
        raw = raw.iloc[0] if not raw.empty else np.nan

    if pd.isna(raw):
        return pd.Series({'date_source': np.nan, 'periode_normalisee': np.nan, 'granularite_temporelle': np.nan, 'annee': np.nan, 'trimestre': np.nan, 'mois': np.nan})

    text = normalize_text(raw)

    m_q = re.fullmatch(r'(\d{4})\s*[Qq]\s*([1-4])', text)
    if m_q:
        year = int(m_q.group(1))
        q = int(m_q.group(2))
        return pd.Series({
            'date_source': text,
            'periode_normalisee': f'{year} T{q}',
            'granularite_temporelle': 'trimestrielle',
            'annee': year,
            'trimestre': q,
            'mois': np.nan,
        })

    m_m = re.fullmatch(r'(\d{4})\s*[Mm]\s*(0?[1-9]|1[0-2])', text)
    if m_m:
        year = int(m_m.group(1))
        month = int(m_m.group(2))
        return pd.Series({
            'date_source': text,
            'periode_normalisee': f"{FRENCH_MONTHS[month]} {year}",
            'granularite_temporelle': 'mensuelle',
            'annee': year,
            'trimestre': int((month - 1) / 3) + 1,
            'mois': month,
        })

    m_year = re.fullmatch(r'(\d{4})', text)
    if m_year:
        year = int(m_year.group(1))
        return pd.Series({
            'date_source': text,
            'periode_normalisee': str(year),
            'granularite_temporelle': 'annuelle',
            'annee': year,
            'trimestre': np.nan,
            'mois': np.nan,
        })

    m_iso = re.fullmatch(r'(\d{4})-(\d{2})-(\d{2})', text)
    if m_iso:
        year = int(m_iso.group(1))
        month = int(m_iso.group(2))
        if 1 <= month <= 12:
            return pd.Series({
                'date_source': text,
                'periode_normalisee': f"{FRENCH_MONTHS[month]} {year}",
                'granularite_temporelle': 'mensuelle',
                'annee': year,
                'trimestre': int((month - 1) / 3) + 1,
                'mois': month,
            })

    return pd.Series({
        'date_source': text,
        'periode_normalisee': text,
        'granularite_temporelle': 'autre',
        'annee': np.nan,
        'trimestre': np.nan,
        'mois': np.nan,
    })

def normalize_time_column(df, date_col='date'):
    if date_col not in df.columns:
        df['date_source'] = np.nan
        df['periode_normalisee'] = np.nan
        df['granularite_temporelle'] = np.nan
        df['annee'] = np.nan
        df['trimestre'] = np.nan
        df['mois'] = np.nan
        return df

    series = df[date_col]
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]

    parsed = series.apply(parse_time_value)
    parsed['date_source'] = parsed['periode_normalisee']
    for c in parsed.columns:
        df[c] = parsed[c]
    df['date'] = df['periode_normalisee']
    return df

def prune_columns(df):
    empty_cols = [c for c in df.columns if df[c].isna().all()]
    by_name = [c for c in df.columns if c in UNNECESSARY_COLUMNS]
    by_prefix = [c for c in df.columns if any(c.startswith(prefix) for prefix in UNNECESSARY_PREFIXES)]
    to_drop = sorted(set(empty_cols + by_name + by_prefix))
    if to_drop:
        df = df.drop(columns=to_drop)
    return df, to_drop

def harmonize_frame(df, source_file, domaine):
    df = df.copy()
    df.columns = normalize_columns(df.columns)
    df = rename_columns_fr(df)
    df, dropped_rows = drop_fully_empty_rows(df)

    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == object:
            df[col] = df[col].map(normalize_text)
            df[col] = df[col].map(translate_text_fr)

    df = translate_values_by_column(df)

    if 'value' in df.columns:
        df['valeur'] = pd.to_numeric(df['value'], errors='coerce')
    elif 'price' in df.columns:
        df['valeur'] = pd.to_numeric(df['price'], errors='coerce')
    else:
        df['valeur'] = np.nan

    date_col = 'date' if 'date' in df.columns else ('dates' if 'dates' in df.columns else None)
    if date_col is not None:
        df = normalize_time_column(df, date_col=date_col)
    else:
        df = normalize_time_column(df, date_col='date')

    df['source_fichier'] = source_file
    df['domaine'] = domaine

    df, dropped_columns = prune_columns(df)

    return df, dropped_rows, dropped_columns

def missing_report(df):
    return (df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame('pct_manquants')

In [ ]:
action_logs = []
exports = []
tables_harmonisees = {}

for group_name, group_path in RAW_GROUPS.items():
    frames = []
    csv_files = sorted(group_path.glob('*.csv'))

    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        df_h, dropped_rows, dropped_columns = harmonize_frame(df, csv_file.name, group_name)

        out_name = csv_file.stem.strip().replace(' ', '_')
        out_path = OUT_DIRS[group_name] / f"{out_name}_nettoye.csv"
        df_h.to_csv(out_path, index=False)

        frames.append(df_h)
        exports.append(str(out_path.relative_to(ROOT)))
        action_logs.append({
            'groupe': group_name,
            'fichier': csv_file.name,
            'lignes': int(len(df_h)),
            'lignes_vides_supprimees': int(dropped_rows),
            'colonnes': int(df_h.shape[1]),
            'colonnes_supprimees': int(len(dropped_columns)),
        })

    if frames:
        concat_df = pd.concat(frames, ignore_index=True)
        concat_df, _ = prune_columns(concat_df)

        concat_path = OUT_DIRS[group_name] / f"{group_name}_harmonise_global.csv"
        concat_df.to_csv(concat_path, index=False)
        exports.append(str(concat_path.relative_to(ROOT)))

        indic_col = 'indicateur' if 'indicateur' in concat_df.columns else None
        if indic_col is None:
            concat_df['indicateur'] = np.nan
            indic_col = 'indicateur'

        annuel = (
            concat_df.groupby(['annee', indic_col], dropna=False, as_index=False)['valeur']
            .mean()
            .rename(columns={'valeur': 'valeur_moyenne', indic_col: 'indicateur'})
            .sort_values(['annee', 'indicateur'])
        )
        annuel_path = OUT_DIRS[group_name] / f"{group_name}_fusion_annuelle.csv"
        annuel.to_csv(annuel_path, index=False)
        exports.append(str(annuel_path.relative_to(ROOT)))

        tables_harmonisees[group_name] = concat_df

actions_df = pd.DataFrame(action_logs)
actions_df

In [ ]:
print('Verification regles temporelles Q -> T et M -> mois')
for group_name, df in tables_harmonisees.items():
    echantillon_q = df[df['date_source'].astype(str).str.contains(r'Q', na=False)][['date_source', 'periode_normalisee']].drop_duplicates().head(5)
    echantillon_m = df[df['date_source'].astype(str).str.contains(r'M', na=False)][['date_source', 'periode_normalisee']].drop_duplicates().head(5)
    print(f'\n[{group_name}]')
    if not echantillon_q.empty:
        print('Exemples trimestre:')
        print(echantillon_q.to_string(index=False))
    if not echantillon_m.empty:
        print('Exemples mois:')
        print(echantillon_m.to_string(index=False))

In [ ]:
for group_name, df in tables_harmonisees.items():
    print(f'\n### Qualite - {group_name}')
    print('Dimensions:', df.shape)
    print('Apercu:')
    print(df.head(3).to_string(index=False))
    print('Manquants (%):')
    print(missing_report(df).head(12).to_string())

    num_cols = [c for c in ['valeur'] if c in df.columns]
    if num_cols:
        print('Description statistique:')
        print(df[num_cols].describe().to_string())

In [ ]:
economy_all = pd.concat(list(tables_harmonisees.values()), ignore_index=True)
economy_all.to_csv(OUT_ROOT / 'economie_harmonise_global.csv', index=False)

synthese = {
    'date_generation': pd.Timestamp.now().isoformat(),
    'nombre_groupes': len(tables_harmonisees),
    'nombre_fichiers_exportes': len(exports),
    'exports': exports,
    'actions': action_logs,
}

(OUT_ROOT / 'synthese_preparation_economie.json').write_text(json.dumps(synthese, ensure_ascii=False, indent=2), encoding='utf-8')

print('Export principal:', (OUT_ROOT / 'economie_harmonise_global.csv').relative_to(ROOT))
print('Synthese:', (OUT_ROOT / 'synthese_preparation_economie.json').relative_to(ROOT))

## Recapitulatif

Ce notebook :
- charge les donnees brutes Economie par sous-domaine
- supprime les lignes entierement vides
- harmonise les colonnes et types
- applique la normalisation temporelle demandee (`Q` -> `T`, `M` -> mois en francais)
- produit des sorties nettoyees par fichier et par sous-domaine
- genere une fusion annuelle et une synthese JSON de traçabilite